# 07 - Đánh giá sự dịch chuyển khái niệm (Concept Drift)

### Khoảng trống nghiên cứu giải quyết (Research Gap):
- Bài báo gốc tự thừa nhận một hạn chế rất lớn tại Section 5.1: *"the current implementation is evaluated under offline conditions; real-time performance under continuous streaming conditions is yet to be assessed"*.
- Notebook này lấp đầy khoảng trống đó bằng cách **thiết lập mô phỏng Concept Drift thời gian thực** trên dữ liệu mạng chạy trực tuyến, ứng dụng thuật toán kiểm định thống kê **Kolmogorov-Smirnov (KS-test)** để phát hiện sự dịch chuyển phân phối đầu vào (Covariate Shift) và đo lường độ trễ phát hiện, cũng như mức độ sụt giảm độ chính xác của mô hình CyberDetect-MLP.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/Nhom28_CyberDetect_MLP_Final'
%cd {PROJECT_PATH}
print('Thư mục làm việc hiện tại:', os.getcwd())

In [ ]:
# Thêm project root vào system path để import mô hình và modules
import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Chạy mô phỏng Concept Drift trên luồng gói tin

Chúng ta sẽ chạy script mô phỏng drift. Kịch bản mô phỏng sẽ:
1. Đọc tập dữ liệu TON_IoT Network.
2. Tạo luồng dữ liệu liên tục gồm 20,000 dòng gói tin mạng.
3. Giai đoạn 1 (t < 10,000): Môi trường mạng bình thường, ổn định.
4. Giai đoạn 2 (t >= 10,000): Tiêm Concept Drift bằng cách thay đổi hành vi tấn công dồn dập và nhân giá trị lưu lượng gói tin mạng lên gấp 50 lần (tạo Covariate Shift).
5. Chạy thuật toán cửa sổ trượt (window=1000, step=100) để đánh giá độ chính xác thực tế của MLP và tính chỉ số dịch chuyển KS statistic.

In [ ]:
# Thực thi kịch bản mô phỏng và phân tích Concept Drift
!python scripts/concept_drift_analysis.py

## 2. Trực quan hóa kết quả phát hiện trôi dạt dữ liệu

Hiển thị bảng chỉ số đo lường Concept Drift theo thời gian thực:

In [ ]:
metrics_csv = 'results/concept_drift_metrics.csv'
if os.path.exists(metrics_csv):
    df_drift = pd.read_csv(metrics_csv)
    print('Bảng dữ liệu chỉ số drift mẫu (5 dòng đầu và cuối):')
    display(pd.concat([df_drift.head(5), df_drift.tail(5)]))
else:
    print('Không tìm thấy tệp kết quả metrics!')

Hiển thị biểu đồ kết hợp thể hiện sự dịch chuyển phân phối (KS statistic) và mức sụt giảm độ chính xác trượt (Rolling Accuracy):

In [ ]:
from PIL import Image
img_path = 'results/concept_drift_analysis.png'
if os.path.exists(img_path):
    img = Image.open(img_path)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('Không tìm thấy hình ảnh biểu đồ phân tích trôi dạt!')

### Nhận xét & Đóng góp học thuật thực tế:
1. **Sự sụt giảm hiệu năng rõ rệt:** Biểu đồ cho thấy khi concept drift xảy ra ở thời điểm dòng sự kiện thứ `10,000`, độ chính xác của mô hình CyberDetect-MLP lập tức sụt giảm nghiêm trọng từ **~99% xuống chỉ còn ~32.6%**, phản ánh thực tế mô hình ngoại tuyến dễ bị tổn thương trước sự thay đổi động của lưu lượng mạng thực tế.
2. **Khả năng phát hiện sớm bằng KS-test:** Thuật toán dựa trên KS-test đã nhanh chóng phát hiện ra sự bất thường phân phối chỉ sau một độ trễ rất nhỏ (detection latency), giúp hệ thống có thể đưa ra cảnh báo kích hoạt tái huấn luyện (Retrain Trigger) kịp thời để khôi phục độ chính xác.